In [1]:
#Task 1: Fit multiple linear regression to predict mobile phone price
import pandas as pd
from sklearn.linear_model import LinearRegression

ph = pd.read_csv("mobile_prices.csv")
feats = ["ram", "storage", "camera_mp", "battery"]

X = ph[feats]
y = ph["price"]

model = LinearRegression()
model.fit(X, y)
print("Model fitted using:", feats)
print("R2 score:", round(model.score(X, y), 3))

Model fitted using: ['ram', 'storage', 'camera_mp', 'battery']
R2 score: 0.927


In [2]:
#Task 2: Show coefficients and intercept with a short comment
print("Intercept:", round(model.intercept_, 2))
for i in range(len(feats)):
    print(feats[i], "->", round(model.coef_[i], 2))

# ram: price goes up by about Rs 1320 for every extra 1 GB of RAM
# storage: price goes up by about Rs 21 for every extra 1 GB of storage
# camera_mp: price goes up by about Rs 74 for every extra 1 MP of camera
# battery: price goes up by about Rs 1 for every extra 1 mAh of battery
# intercept: the base price when all features are 0, it has no real meaning for a phone

Intercept: 1388.95
ram -> 1320.49
storage -> 20.77
camera_mp -> 74.48
battery -> 1.17


In [3]:
#Task 3: Calculate VIF for each feature
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

Xc = add_constant(X)
for i in range(1, Xc.shape[1]):
    print(Xc.columns[i], "VIF:", round(variance_inflation_factor(Xc.values, i), 2))

# ram and storage have VIF above 5 (about 7), so they show multicollinearity. camera_mp and battery are close to 1, so they are fine.

ram VIF: 7.01
storage VIF: 6.99
camera_mp VIF: 1.01
battery VIF: 1.01


In [4]:
#Task 4: Find the most influential feature
# raw coefficients depend on units, so multiply each by the feature std to compare fairly
for i in range(len(feats)):
    print(feats[i], "-> impact:", round(model.coef_[i] * X[feats[i]].std(), 2))

# ram is the most influential feature. After adjusting for units, a typical change in ram moves the price the most, more than storage, camera or battery.
# This makes sense because RAM is usually the main thing that decides the price of a phone.

ram -> impact: 4276.17
storage -> impact: 1921.89
camera_mp -> impact: 1588.08
battery -> impact: 1027.24


In [5]:
#Task 5: Remove one collinear feature and re-fit the model
from sklearn.metrics import mean_squared_error

feats2 = ["ram", "camera_mp", "battery"]
model2 = LinearRegression()
model2.fit(ph[feats2], y)

print("Old model:")
print("R2:", round(model.score(X, y), 3), "MSE:", round(mean_squared_error(y, model.predict(X)), 0))
print("New model (storage removed):")
print("R2:", round(model2.score(ph[feats2], y), 3), "MSE:", round(mean_squared_error(y, model2.predict(ph[feats2])), 0))

print("\nNew intercept:", round(model2.intercept_, 2))
for i in range(len(feats2)):
    print(feats2[i], "->", round(model2.coef_[i], 2))

# After removing storage, R2 dropped only a little (0.927 to 0.914) and MSE went up a bit, so the fit is almost the same.
# The ram coefficient changed from about 1320 to 1871 because it now carries the effect that storage was sharing with it, so the model is easier to read.
# camera_mp and battery coefficients stayed almost the same.

Old model:
R2: 0.927 MSE: 3093126.0
New model (storage removed):
R2: 0.914 MSE: 3616499.0

New intercept: 761.84
ram -> 1870.69
camera_mp -> 76.3
battery -> 1.22
